# 05 — Neural ODE

**What it does**: Learns a vector field `dz/dt = f(z)` from observed trajectories.
The dynamics function f is parameterized as a neural network, and we integrate it
forward with a standard numerical integrator (RK4 here).

**Architecture spec** (from problem statement):
- Vector field MLP: 2 hidden layers × 64 units, tanh
- RK4 fixed-step integrator (backpropagating through integrator steps)
- 200 trajectories, t in [0, 10] at dt=0.05
- Train on t in [0, 5], test rollout to t=10 and beyond

**Key insight**: Unlike PINN, the Neural ODE doesn't need to know the equation.
It learns the dynamics from data. But it can only extrapolate if the learned
vector field is accurate beyond the training regime.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Neural ODE using device: {device}')

In [ ]:
# ============================================================
# Ground Truth: Exact Damped Oscillator Solution
# ============================================================

def oscillator_exact(t, z0=1.0, v0=0.0, omega0=2.0, zeta=0.1):
    """Exact underdamped oscillator position at times t."""
    omega_d = omega0 * math.sqrt(1 - zeta**2)
    C1 = z0
    C2 = (v0 + zeta * omega0 * z0) / omega_d
    return torch.exp(-zeta * omega0 * t) * (
        C1 * torch.cos(omega_d * t) + C2 * torch.sin(omega_d * t)
    )

def oscillator_exact_velocity(t, z0=1.0, v0=0.0, omega0=2.0, zeta=0.1):
    """Exact velocity dz/dt at times t."""
    omega_d = omega0 * math.sqrt(1 - zeta**2)
    C1 = z0
    C2 = (v0 + zeta * omega0 * z0) / omega_d
    decay   = torch.exp(-zeta * omega0 * t)
    pos_der = (-zeta * omega0 * (C1*torch.cos(omega_d*t) + C2*torch.sin(omega_d*t)) +
               omega_d * (-C1*torch.sin(omega_d*t) + C2*torch.cos(omega_d*t)))
    return decay * pos_der

In [ ]:
# ============================================================
# Dataset Generation
# ============================================================

def generate_oscillator_dataset(
    n_traj=200, t_max=10.0, dt=0.05,
    omega0=2.0, zeta=0.1,
    z0_range=(-1.0, 1.0), v0_range=(-1.0, 1.0),
    seed=42
):
    """
    Generate n_traj trajectories from random initial conditions.

    Returns:
        t_grid: (n_steps,) time points
        states: (n_traj, n_steps, 2) trajectories [z, dz/dt]
        ics: (n_traj, 2) initial conditions
    """
    torch.manual_seed(seed)
    t_grid = torch.arange(0, t_max + dt, dt)
    n_steps = len(t_grid)

    z0_vals = torch.rand(n_traj) * (z0_range[1] - z0_range[0]) + z0_range[0]
    v0_vals = torch.rand(n_traj) * (v0_range[1] - v0_range[0]) + v0_range[0]

    states = torch.zeros(n_traj, n_steps, 2)

    for i in range(n_traj):
        z0 = z0_vals[i].item()
        v0 = v0_vals[i].item()
        z_traj = oscillator_exact(t_grid, z0=z0, v0=v0, omega0=omega0, zeta=zeta)
        v_traj = oscillator_exact_velocity(t_grid, z0=z0, v0=v0, omega0=omega0, zeta=zeta)
        states[i, :, 0] = z_traj
        states[i, :, 1] = v_traj

    ics = states[:, 0, :]  # (n_traj, 2)
    return t_grid, states, ics


# Generate 200 training trajectories
t_grid, all_states, all_ics = generate_oscillator_dataset(n_traj=200, seed=42)
print(f'Dataset shape: {all_states.shape}')  # (200, 201, 2)

# Split: train = first 100 time steps (t <= 5), test = all 200 steps
train_end_idx = int(5.0 / 0.05) + 1  # index of t=5
t_train = t_grid[:train_end_idx]
t_full  = t_grid
print(f'Train time range: 0 to {t_train[-1]:.2f}, {len(t_train)} steps')
print(f'Full time range:  0 to {t_full[-1]:.2f}, {len(t_full)} steps')

# Visualize some training trajectories
plt.figure(figsize=(10, 4))
for i in range(5):
    plt.plot(t_grid.numpy(), all_states[i, :, 0].numpy(), alpha=0.7)
plt.axvspan(5, t_grid[-1].item(), alpha=0.1, color='red')
plt.axvline(x=5, color='red', linestyle='--', label='Train/Test split')
plt.title('Sample Oscillator Trajectories')
plt.xlabel('t'); plt.ylabel('z(t)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Neural ODE Architecture
# ============================================================

class ODEFunc(nn.Module):
    """
    Parameterizes the vector field dz/dt = f_theta(z).

    Input:  state z = [position, velocity]  (dim=2)
    Output: d/dt z = [velocity, acceleration]  (dim=2)

    Spec: 2 hidden layers x 64 units, tanh activation.
    """

    def __init__(self, state_dim=2, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, state_dim)
        )

    def forward(self, z):
        return self.net(z)


# ============================================================
# RK4 Integrator (fixed step, differentiable)
# ============================================================

def rk4_step(f, z, dt):
    """One RK4 step: z(t+dt) = z(t) + (k1 + 2*k2 + 2*k3 + k4) / 6."""
    k1 = f(z)
    k2 = f(z + dt/2 * k1)
    k3 = f(z + dt/2 * k2)
    k4 = f(z + dt   * k3)
    return z + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


def odeint_rk4(f, z0, t_span, dt=0.05):
    """
    Integrate dz/dt = f(z) from t_span[0] to t_span[-1] using RK4.

    Args:
        f: ODE function (differentiable)
        z0: (batch, state_dim) initial state
        t_span: 1D tensor of time points (uniformly spaced assumed)
        dt: step size

    Returns:
        trajectory: (batch, len(t_span), state_dim)
    """
    traj = [z0]
    z = z0
    for i in range(1, len(t_span)):
        z = rk4_step(f, z, dt)
        traj.append(z)
    return torch.stack(traj, dim=1)  # (batch, T, state_dim)


n_params = sum(p.numel() for p in ODEFunc().parameters())
print(f'ODEFunc parameters: {n_params}')

In [ ]:
# ============================================================
# Training
# ============================================================

def train_neural_ode(
    all_ics, all_states, t_train,
    epochs=1000, lr=1e-3, batch_size=32, dt=0.05
):
    """
    Train ODEFunc by minimizing MSE between predicted and true trajectories
    over the training time window.

    Args:
        all_ics: (n_traj, 2) initial conditions
        all_states: (n_traj, T_full, 2) true trajectories
        t_train: 1D tensor of training time points

    Returns:
        model, history
    """
    n_traj = all_ics.shape[0]
    n_train_steps = len(t_train)

    model = ODEFunc().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)

    history = []
    print(f'Training Neural ODE: {epochs} epochs')
    t_start = time.time()

    for epoch in range(1, epochs + 1):
        # Sample a random mini-batch of trajectories
        idx = torch.randperm(n_traj)[:batch_size]
        z0_batch  = all_ics[idx].to(device)         # (B, 2)
        z_gt_batch = all_states[idx, :n_train_steps, :].to(device)  # (B, T, 2)

        optimizer.zero_grad()
        z_pred = odeint_rk4(model, z0_batch, t_train, dt=dt)  # (B, T, 2)
        loss = torch.mean((z_pred - z_gt_batch) ** 2)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        history.append(loss.item())
        if epoch % 200 == 0 or epoch == 1:
            print(f'  Epoch {epoch:5d}/{epochs} | MSE={loss.item():.6f}')

    elapsed = time.time() - t_start
    print(f'Done in {elapsed:.1f}s')
    model.eval()
    return model, history


node_model, node_history = train_neural_ode(
    all_ics, all_states, t_train, epochs=2000
)

plt.figure(figsize=(8, 4))
plt.semilogy(node_history)
plt.title('Neural ODE Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.grid(True, which='both', alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Evaluation: In-window and Rollout
# ============================================================

node_model.eval()

# Select a few test trajectories (unseen initial conditions)
test_ics = all_ics[-5:].to(device)  # last 5 trajectories
test_gt  = all_states[-5:]           # (5, T_full, 2)

with torch.no_grad():
    z_pred_full = odeint_rk4(node_model, test_ics, t_full, dt=0.05).cpu()

# Compute L2 error at each time step
errors_by_time = torch.norm(z_pred_full - test_gt, dim=2).mean(dim=0)  # (T_full,)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Individual trajectory plots
for col_idx, traj_idx in enumerate([0, 2, 4]):
    ax = axes[0, col_idx]
    ax.plot(t_full.numpy(), test_gt[traj_idx, :, 0].numpy(), label='Exact', linewidth=2)
    ax.plot(t_full.numpy(), z_pred_full[traj_idx, :, 0].numpy(),
            label='Neural ODE', linestyle='--', linewidth=2)
    ax.axvspan(5, t_full[-1].item(), alpha=0.1, color='red')
    ax.axvline(x=5, color='red', linestyle=':', linewidth=1.5)
    ax.set_title(f'Trajectory {traj_idx+1}')
    ax.set_xlabel('t'); ax.set_ylabel('z(t)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Error over time
ax = axes[1, 0]
ax.semilogy(t_full.numpy(), errors_by_time.numpy())
ax.axvline(x=5, color='red', linestyle='--', label='Train boundary')
ax.set_title('Mean L2 Error vs Time')
ax.set_xlabel('t'); ax.set_ylabel('|z_pred - z_exact|')
ax.legend(); ax.grid(True, which='both', alpha=0.3)

# Phase portrait
ax = axes[1, 1]
for traj_idx in range(5):
    ax.plot(test_gt[traj_idx, :, 0].numpy(), test_gt[traj_idx, :, 1].numpy(),
            color='blue', alpha=0.5)
    ax.plot(z_pred_full[traj_idx, :, 0].numpy(), z_pred_full[traj_idx, :, 1].numpy(),
            color='red', alpha=0.5, linestyle='--')
ax.set_title('Phase Portrait (blue=exact, red=NODE)')
ax.set_xlabel('z'); ax.set_ylabel('dz/dt'); ax.grid(True, alpha=0.3)

# Summary stats
train_err = errors_by_time[:len(t_train)].mean().item()
test_err  = errors_by_time[len(t_train):].mean().item()
axes[1, 2].bar(['Train window\n(t in 0-5)', 'Test rollout\n(t in 5-10)'],
               [train_err, test_err],
               color=['green', 'red'], alpha=0.7)
axes[1, 2].set_title('Mean Error: Train vs Test')
axes[1, 2].set_ylabel('Mean L2 Error'); axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Neural ODE: In-window Fit and Rollout')
plt.tight_layout(); plt.show()

print(f'Mean L2 error (train window t=0-5): {train_err:.6f}')
print(f'Mean L2 error (rollout   t=5-10 ): {test_err:.6f}')
print(f'Error ratio rollout/train: {test_err / (train_err + 1e-10):.2f}x')

In [ ]:
# ============================================================
# Visualize Learned vs True Vector Field
# ============================================================

# True vector field: f(z, v) = [v, -2*zeta*omega0*v - omega0^2*z]
def true_vf(z, v, omega0=2.0, zeta=0.1):
    dz = v
    dv = -2*zeta*omega0*v - omega0**2 * z
    return dz, dv

z_grid  = torch.linspace(-1.5, 1.5, 20)
v_grid  = torch.linspace(-3.0, 3.0, 20)
ZZ, VV = torch.meshgrid(z_grid, v_grid, indexing='ij')

states_grid = torch.stack([ZZ.flatten(), VV.flatten()], dim=1).to(device)

node_model.eval()
with torch.no_grad():
    learned_dstate = node_model(states_grid).cpu()

true_dz, true_dv = true_vf(ZZ.flatten(), VV.flatten())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (Udz, Udv, title) in zip(
    axes,
    [(true_dz.reshape(20,20), true_dv.reshape(20,20), 'True Vector Field'),
     (learned_dstate[:,0].reshape(20,20), learned_dstate[:,1].reshape(20,20), 'Learned Vector Field')]
):
    ax.quiver(ZZ.numpy(), VV.numpy(), Udz.numpy(), Udv.numpy(),
              angles='xy', scale_units='xy', alpha=0.7)
    ax.set_title(title); ax.set_xlabel('z'); ax.set_ylabel('dz/dt')
    ax.grid(True, alpha=0.3)

plt.suptitle('Neural ODE: True vs Learned Dynamics')
plt.tight_layout(); plt.show()